# Tag selection via event filtering (notebook)

This notebook builds a **curated Polymarket tag allowlist** in a way that avoids the main failure mode you saw (sports/entertainment tags dominating because of ambiguous tokens like `sec`).

Flow:
1) Sample active events from Gamma.
2) Filter out obvious sports/entertainment events.
3) Identify **signal-family-relevant events** by matching event titles against your family keyword rules.
4) From those relevant events, count tags by frequency **per family**.
5) Build an allowlist = union(top tags per family), apply a small denylist.
6) Ingest markets from the allowlist into Postgres.
7) Run family matching on ingested markets and inspect coverage.


In [18]:
# Cell 0 — Imports + config

import os
import re
from collections import Counter, defaultdict

import pandas as pd

from polyscanner.env import load_env
from polyscanner.ingestion.pm_markets import ingest_markets_from_tag_ids
from polyscanner.ingestion.tag_base import iter_active_events
from polyscanner.pipeline.signal_family_mvp import match_market_to_rule
from polyscanner.signal_family_rules import RULES_BY_SLUG

load_env()

BASE_URL = os.getenv("POLYMARKET_API_BASE_URL") or "https://gamma-api.polymarket.com"
DB_URL = os.getenv("DATABASE_URL")
assert DB_URL, "Missing DATABASE_URL"

# Sampling
EVENT_SAMPLE_TARGET = 10000   # 500–2000
EVENTS_PAGE_SIZE = 2000
EVENTS_MAX_PAGES = 300        # upper bound; we stop at EVENT_SAMPLE_TARGET
SLEEP_S = 0.0002

# Allowlist construction
TOP_TAGS_PER_FAMILY = 1500

# Ingestion
MARKETS_CAP_PER_TAG = 50
TAG_EVENTS_MAX_PAGES = 3     # tag_id -> /events pagination depth
TAG_EVENTS_PAGE_SIZE = 50


In [19]:
# Cell 1 — Small helper methods (single responsibility)

def _norm(s: str) -> str:
    """Lowercase + whitespace normalize for keyword matching."""
    return re.sub(r"\s+", " ", (s or "").lower()).strip()


def event_title(ev: dict) -> str:
    """Return canonical event text used for filtering/matching."""
    return str(ev.get("title") or ev.get("question") or "").strip()


def extract_event_tags(ev: dict) -> list[dict]:
    """Return tag dicts from an event payload (best-effort)."""
    tags = ev.get("tags") or []
    if not isinstance(tags, list):
        return []
    out = []
    for t in tags:
        if not isinstance(t, dict):
            continue
        if t.get("id") is None:
            continue
        out.append(t)
    return out


# Lightweight event-type filter (avoid obvious sports/entertainment)
SPORTS_TERMS = {
    "regular season",
    "playoffs",
    "championship",
    "final",
    "semifinal",
    "win the",
    "wins the",
    "vs",
    "versus",
    "team",
    "teams",
    "women's",
    "men's",
    "nba",
    "nfl",
    "nhl",
    "mlb",
    "ufc",
    "ncaa",
    "wnba",
    "mls",
    "soccer",
    "football",
    "basketball",
    "baseball",
    "hockey",
    "tennis",
    "golf",
}

ENT_TERMS = {
    "movie",
    "movies",
    "oscars",
    "grammys",
    "tv",
    "episode",
    "album",
    "song",
    "celebrity",
    "actor",
    "actress",
}


def is_sports_or_entertainment_event(ev: dict) -> bool:
    """Return True if the event title looks like sports/entertainment."""
    txt = _norm(event_title(ev))
    if not txt:
        return False
    if any(t in txt for t in SPORTS_TERMS):
        return True
    if any(t in txt for t in ENT_TERMS):
        return True
    return False


def event_family_matches(ev: dict) -> list[str]:
    """Return list of family slugs whose rules match the event title."""
    txt = event_title(ev)
    if not txt:
        return []
    matched = []
    for slug, rule in RULES_BY_SLUG.items():
        score, _terms = match_market_to_rule(question=txt, category=None, rule=rule)
        if score > 0:
            matched.append(slug)
    return matched


In [20]:
# Cell 2 — Sample active events

events = []
for ev in iter_active_events(
    base_url=BASE_URL,
    limit=EVENTS_PAGE_SIZE,
    max_pages=EVENTS_MAX_PAGES,
    sleep_s=SLEEP_S,
):
    if isinstance(ev, dict):
        events.append(ev)
    if len(events) >= EVENT_SAMPLE_TARGET:
        break

len(events), event_title(events[0])


(2000, 'MicroStrategy sells any Bitcoin by ___ ?')

In [21]:
# Cell 3 — Filter sports/entertainment events

kept = [ev for ev in events if not is_sports_or_entertainment_event(ev)]
dropped = [ev for ev in events if is_sports_or_entertainment_event(ev)]

len(events), len(kept), len(dropped), event_title(kept[0])


(2000, 1337, 663, 'MicroStrategy sells any Bitcoin by ___ ?')

In [22]:
# Cell 4 — Keep only events that match at least one signal family (title-based)

kept_relevant = []
event_families = {}  # python-object id(event) -> [families]

for ev in kept:
    fams = event_family_matches(ev)
    if fams:
        kept_relevant.append(ev)
        event_families[id(ev)] = fams

# Coverage diagnostics: how many events match each family?
fam_event_counts = Counter()
for fams in event_families.values():
    for f in fams:
        fam_event_counts[f] += 1

len(kept), len(kept_relevant), pd.Series(dict(fam_event_counts)).sort_values(ascending=False)


(1337,
 23,
 fomc_surprises                           16
 taiwan_geopolitical_risk                  3
 real_yields_long_rates                    2
 datacenter_power_grid_constraints         1
 it_spending_cycle_enterprise_cloud_ai     1
 dtype: int64)

In [23]:
# Cell 5 — Count tags per family from the relevant events

family_tag_counts = {slug: Counter() for slug in RULES_BY_SLUG.keys()}
tag_meta = {}  # tag_id -> (label, slug)

for ev in kept_relevant:
    fams = event_families.get(id(ev), [])
    for t in extract_event_tags(ev):
        try:
            tid = int(t.get("id"))
        except Exception:
            continue
        tag_meta.setdefault(tid, (t.get("label"), t.get("slug")))
        for fam in fams:
            family_tag_counts[fam][tid] += 1

# Inspect top tags for one family
FAMILY = "fomc_surprises"  # change me

top = family_tag_counts[FAMILY].most_common(25)
pd.DataFrame([
    {
        "tag_id": tid,
        "count": c,
        "label": tag_meta.get(tid, (None, None))[0],
        "slug": tag_meta.get(tid, (None, None))[1],
    }
    for tid, c in top
])


,tag_id,count,label,slug
0,159,15,Fed,fed
1,100196,12,Fed Rates,fed-rates
2,2,10,Politics,politics
3,100328,10,Economy,economy
4,101550,8,Jerome Powell,jerome-powell
5,126,6,Trump,trump
6,101800,6,Economic Policy,economic-policy
7,120,4,Finance,finance
8,101191,2,Trump Presidency,trump-presidency
9,103123,2,Parent For Derivative,parent-for-derivative


In [24]:
# Cell 6 — Build allowlist = union(top tags per family, with a small denylist

DENY_SLUGS = {
    # too broad
    "politics",
    "elections",
    "primaries",
    "world",
    "economy",
    "business",
    "finance",
    "sports",
    "soccer",
    "culture",
    "pop-culture",
    # containers/templates
    "recurring",
    "up-or-down",
}

allow = set()
selected_by_family = {}

for fam, cnts in family_tag_counts.items():
    rows = []
    n = 0
    for tid, c in cnts.most_common(400):
        label, slug = tag_meta.get(tid, (None, None))
        slug_l = str(slug or "").strip().lower()
        if slug_l in DENY_SLUGS:
            continue
        rows.append({"tag_id": int(tid), "count": int(c), "label": label, "slug": slug})
        allow.add(int(tid))
        n += 1
        if n >= TOP_TAGS_PER_FAMILY:
            break
    selected_by_family[fam] = rows

allowlist_tag_ids = sorted(allow)

len(allowlist_tag_ids), allowlist_tag_ids[:30], {k: len(v) for k, v in selected_by_family.items()}


(34,
 [126,
  129,
  159,
  175,
  303,
  514,
  824,
  867,
  1101,
  1401,
  1563,
  1597,
  100196,
  100199,
  100265,
  100478,
  101191,
  101337,
  101550,
  101794,
  101800,
  101999,
  102028,
  102289,
  102458,
  102957,
  103123,
  103338,
  103339,
  103340],
 {'fomc_surprises': 19,
  'real_yields_long_rates': 4,
  'us_china_semis_export_controls': 0,
  'taiwan_geopolitical_risk': 6,
  'crypto_regime_changes': 0,
  'ai_regulation_big_tech_enforcement': 0,
  'datacenter_power_grid_constraints': 4,
  'antitrust_platforms_app_stores_ads': 0,
  'it_spending_cycle_enterprise_cloud_ai': 5,
  'healthcare_policy_reimbursement': 0,
  'consumer_credit_conditions_cycle': 0})

In [25]:
# Cell 7 — Ingest markets from allowlist tags into Postgres

ingest_result = ingest_markets_from_tag_ids(
    db_url=DB_URL,
    base_url=BASE_URL,
    tag_ids=allowlist_tag_ids,
    markets_cap_per_tag=MARKETS_CAP_PER_TAG,
    max_events_pages=TAG_EVENTS_MAX_PAGES,
    events_page_size=TAG_EVENTS_PAGE_SIZE,
    sleep_s=SLEEP_S,
)
ingest_result


{'tags': 34,
 'per_tag_fetched': {126: 50,
  129: 7,
  159: 50,
  175: 5,
  303: 50,
  514: 50,
  824: 2,
  867: 37,
  1101: 50,
  1401: 50,
  1563: 14,
  1597: 50,
  100196: 50,
  100199: 27,
  100265: 50,
  100478: 2,
  101191: 50,
  101337: 50,
  101550: 50,
  101794: 50,
  101800: 50,
  101999: 50,
  102028: 16,
  102289: 50,
  102458: 50,
  102957: 2,
  103123: 50,
  103338: 2,
  103339: 2,
  103340: 1,
  103341: 1,
  103342: 1,
  103358: 5,
  103715: 50},
 'fetched_raw': 1074,
 'unique_markets': 658,
 'normalized': 658,
 'upserted': 658}

In [26]:
# Cell 8 — Fetch recent markets from Postgres

import psycopg


def fetch_recent_pm_markets(db_url: str, limit: int = 5000) -> list[dict]:
    """Load recent pm_market rows for local matching diagnostics."""
    db_url = db_url.strip().replace("postgresql+psycopg://", "postgresql://")
    conn = psycopg.connect(db_url)
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                select pm_market_id, question, category, probability, volume_usd
                from pm_market
                order by last_seen_at desc
                limit %s
                """,
                (int(limit),),
            )
            rows = cur.fetchall()
    finally:
        conn.close()

    out = []
    for pm_market_id, question, category, probability, volume_usd in rows:
        out.append(
            {
                "pm_market_id": int(pm_market_id),
                "question": question,
                "category": category,
                "probability": probability,
                "volume_usd": volume_usd,
            }
        )
    return out


markets = fetch_recent_pm_markets(DB_URL, limit=18000)
len(markets), markets[0]["question"]


(2795, 'Xi Jinping divorce before 2027?')

In [27]:
# Cell 9 — Match markets to families (rule-based) and inspect coverage

matches = []
for m in markets:
    q = str(m.get("question") or "")
    cat = str(m.get("category") or "")
    for slug, rule in RULES_BY_SLUG.items():
        score, terms = match_market_to_rule(question=q, category=cat, rule=rule)
        if score > 0:
            matches.append(
                {
                    "pm_market_id": m["pm_market_id"],
                    "question": q,
                    "family": slug,
                    "match_score": float(score),
                    "matched_terms": terms,
                    "volume_usd": m.get("volume_usd"),
                }
            )

df_matches = pd.DataFrame(matches)
df_matches.shape, df_matches["pm_market_id"].nunique() if not df_matches.empty else 0


((190, 6), 188)

In [28]:
# Cell 10 — Coverage + example matches

if df_matches.empty:
    print("No matches found. Likely: allowlist too broad + rules too strict / missing.")
else:
    display(df_matches.groupby("family")["pm_market_id"].nunique().sort_values(ascending=False))

    FAMILY = "fomc_surprises"  # change me
    display(
        df_matches[df_matches["family"] == FAMILY]
        .sort_values(["match_score", "volume_usd"], ascending=False)
        .head(12)[["pm_market_id", "match_score", "matched_terms", "question"]]
    )


family
fomc_surprises                           149
real_yields_long_rates                    16
consumer_credit_conditions_cycle          11
datacenter_power_grid_constraints          6
taiwan_geopolitical_risk                   6
ai_regulation_big_tech_enforcement         1
it_spending_cycle_enterprise_cloud_ai      1
Name: pm_market_id, dtype: int64

,pm_market_id,match_score,matched_terms,question
103,654412,0.285714,"[fed, interest rate]",Will the Fed decrease interest rates by 50+ bp...
106,654415,0.285714,"[fed, interest rate]",Will the Fed increase interest rates by 25+ bp...
11,572478,0.285714,"[fed, powell]",Will Trump nominate Jerome Powell as the next ...
104,654413,0.285714,"[fed, interest rate]",Will the Fed decrease interest rates by 25 bps...
105,654414,0.285714,"[fed, interest rate]",Will there be no change in Fed interest rates ...
70,616902,0.285714,"[fed, rate cut]",Will no Fed rate cuts happen in 2026?
100,616908,0.285714,"[fed, rate cut]",Will 6 Fed rate cuts happen in 2026?
107,669660,0.285714,"[fed, interest rate]",Will the Fed decrease interest rates by 50+ bp...
102,616914,0.285714,"[fed, rate cut]",Will 12 or more Fed rate cuts happen in 2026?
110,669663,0.285714,"[fed, interest rate]",Will the Fed increase interest rates by 25+ bp...


In [29]:
from pathlib import Path
import pandas as pd

out_dir = Path("notebooks/exports")
out_dir.mkdir(parents=True, exist_ok=True)

# 1) Selected tags per family (dict -> DF)
df_selected_tags = pd.DataFrame(
    [{"family": fam, **row} for fam, rows in selected_by_family.items() for row in rows]
)
df_selected_tags.to_csv(out_dir / "selected_tags_by_family.csv", index=False)

# 2) Allowlist (list -> DF)
pd.DataFrame({"tag_id": allowlist_tag_ids}).to_csv(out_dir / "allowlist_tag_ids.csv", index=False)

# 3) Market-family matches (already a DF)
df_matches.to_csv(out_dir / "market_family_matches.csv", index=False)

pd.Series(dict(fam_event_counts)).rename("event_count").reset_index().rename(columns={"index":"family"}) \
  .to_csv(out_dir / "family_event_counts.csv", index=False)

print("Wrote:")
for p in sorted(out_dir.glob("*.csv")):
    print("-", p)

Wrote:
- notebooks/exports/allowlist_tag_ids.csv
- notebooks/exports/family_event_counts.csv
- notebooks/exports/market_family_matches.csv
- notebooks/exports/selected_tags_by_family.csv


In [30]:
# Broad candidate tags from kept events (no family gate)
tag_counts_all = Counter()
tag_meta_all = {}

for ev in kept:
    for t in extract_event_tags(ev):
        try:
            tid = int(t.get("id"))
        except Exception:
            continue
        tag_counts_all[tid] += 1
        tag_meta_all.setdefault(tid, (t.get("label"), t.get("slug")))

# Keep a large candidate pool so rare-but-relevant tags can appear
CANDIDATE_TAGS_N = 1500
candidate_tag_ids = [tid for tid, _c in tag_counts_all.most_common(CANDIDATE_TAGS_N)]

len(candidate_tag_ids), candidate_tag_ids[:20]

(443,
 [2,
  102169,
  21,
  101757,
  1312,
  102127,
  144,
  102892,
  102786,
  103153,
  100265,
  101970,
  235,
  39,
  101267,
  101312,
  818,
  102467,
  1101,
  126])

In [31]:
from polyscanner.ingestion.tag_selection import market_text
from polyscanner.ingestion.tag_base import fetch_markets_for_tag

TARGET_FAMILIES = [
    "crypto_regime_changes",
    "antitrust_platforms_app_stores_ads",
    "healthcare_policy_reimbursement",
    "us_china_semis_export_controls",
]

def compute_tag_yield_for_targets(tag_id: int, markets: list[dict], targets: list[str]) -> dict:
    counts = {fam: 0 for fam in targets}
    for m in markets:
        q = market_text(m)
        if not q:
            continue
        for fam in targets:
            rule = RULES_BY_SLUG[fam]
            score, terms = match_market_to_rule(question=q, category=None, rule=rule)
            if score > 0:
                counts[fam] += 1
    return {"tag_id": int(tag_id), **counts, "n_markets": len(markets)}

# Sample
MARKETS_PER_TAG = 20
markets_cache2 = {}
rows = []

for tid in candidate_tag_ids:
    mkts = fetch_markets_for_tag(int(tid), markets_cap=MARKETS_PER_TAG, base_url=BASE_URL, max_events_pages=3)
    markets_cache2[int(tid)] = mkts
    rows.append(compute_tag_yield_for_targets(int(tid), mkts, TARGET_FAMILIES))

df_target = pd.DataFrame(rows)
df_target.head()


,tag_id,crypto_regime_changes,antitrust_platforms_app_stores_ads,healthcare_policy_reimbursement,us_china_semis_export_controls,n_markets
0,2,0,0,0,0,20
1,102169,0,0,0,0,20
2,21,0,0,0,0,20
3,101757,0,0,0,0,20
4,1312,0,0,0,0,20


In [32]:
for fam in TARGET_FAMILIES:
    print("\n==", fam, "==")
    top = df_target.sort_values([fam, "n_markets"], ascending=False).head(15)
    display(
        top.merge(
            pd.DataFrame([{"tag_id": tid, "label": tag_meta_all.get(tid,(None,None))[0], "slug": tag_meta_all.get(tid,(None,None))[1]}
                          for tid in top["tag_id"].astype(int).tolist()]),
            on="tag_id",
            how="left",
        )[["tag_id","label","slug",fam,"n_markets"]]
    )

    # Qualitative check: print markets for the best tag
    best_tid = int(top.iloc[0]["tag_id"])
    print("Best tag_id:", best_tid, "slug:", tag_meta_all.get(best_tid,(None,None))[1])
    for q in [market_text(m) for m in markets_cache2[best_tid][:20]]:
        if q:
            print("-", q)



== crypto_regime_changes ==


,tag_id,label,slug,crypto_regime_changes,n_markets
0,2,Politics,politics,0,20
1,102169,Hide From New,hide-from-new,0,20
2,21,Crypto,crypto,0,20
3,101757,Recurring,recurring,0,20
4,1312,Crypto Prices,crypto-prices,0,20
5,102127,Up or Down,up-or-down,0,20
6,144,Elections,elections,0,20
7,102892,5M,5M,0,20
8,102786,Nov 4 Elections,nov-4-elections,0,20
9,103153,"rewards 5, 4.5, 50",rewards-5-4pt5-50,0,20


Best tag_id: 2 slug: politics
- Macron out in 2025?
- Macron out by June 30, 2026?
- Macron out by October 31, 2025?
- Will Trump deport less than 250,000?
- Will Trump deport 250,000-500,000 people?
- Will Trump deport 500,000-750,000- people?
- Will Trump deport 750,000-1,000,000 people?
- Will Trump deport 1,000,000-1,250,000 people?
- Will Trump deport 1,750,000-2,000,000 people?
- Will Trump deport 1,250,000-1,500,000 people?
- Will Trump deport 1,500,000-1,750,000 people?
- Will Trump deport 2,000,000 or more people?
- Will Trump deport 750,000 or more people in 2025?
- China x India military clash by December 31?
- China x India military clash by June 30?
- China x India military clash by December 31, 2026?
- NATO/EU troops fighting in Ukraine in 2025?
- NATO/EU troops fighting in Ukraine in June 30, 2026?
- Starmer out in 2025?
- Starmer out by June 30, 2026?

== antitrust_platforms_app_stores_ads ==


,tag_id,label,slug,antitrust_platforms_app_stores_ads,n_markets
0,2,Politics,politics,0,20
1,102169,Hide From New,hide-from-new,0,20
2,21,Crypto,crypto,0,20
3,101757,Recurring,recurring,0,20
4,1312,Crypto Prices,crypto-prices,0,20
5,102127,Up or Down,up-or-down,0,20
6,144,Elections,elections,0,20
7,102892,5M,5M,0,20
8,102786,Nov 4 Elections,nov-4-elections,0,20
9,103153,"rewards 5, 4.5, 50",rewards-5-4pt5-50,0,20


Best tag_id: 2 slug: politics
- Macron out in 2025?
- Macron out by June 30, 2026?
- Macron out by October 31, 2025?
- Will Trump deport less than 250,000?
- Will Trump deport 250,000-500,000 people?
- Will Trump deport 500,000-750,000- people?
- Will Trump deport 750,000-1,000,000 people?
- Will Trump deport 1,000,000-1,250,000 people?
- Will Trump deport 1,750,000-2,000,000 people?
- Will Trump deport 1,250,000-1,500,000 people?
- Will Trump deport 1,500,000-1,750,000 people?
- Will Trump deport 2,000,000 or more people?
- Will Trump deport 750,000 or more people in 2025?
- China x India military clash by December 31?
- China x India military clash by June 30?
- China x India military clash by December 31, 2026?
- NATO/EU troops fighting in Ukraine in 2025?
- NATO/EU troops fighting in Ukraine in June 30, 2026?
- Starmer out in 2025?
- Starmer out by June 30, 2026?

== healthcare_policy_reimbursement ==


,tag_id,label,slug,healthcare_policy_reimbursement,n_markets
0,2,Politics,politics,0,20
1,102169,Hide From New,hide-from-new,0,20
2,21,Crypto,crypto,0,20
3,101757,Recurring,recurring,0,20
4,1312,Crypto Prices,crypto-prices,0,20
5,102127,Up or Down,up-or-down,0,20
6,144,Elections,elections,0,20
7,102892,5M,5M,0,20
8,102786,Nov 4 Elections,nov-4-elections,0,20
9,103153,"rewards 5, 4.5, 50",rewards-5-4pt5-50,0,20


Best tag_id: 2 slug: politics
- Macron out in 2025?
- Macron out by June 30, 2026?
- Macron out by October 31, 2025?
- Will Trump deport less than 250,000?
- Will Trump deport 250,000-500,000 people?
- Will Trump deport 500,000-750,000- people?
- Will Trump deport 750,000-1,000,000 people?
- Will Trump deport 1,000,000-1,250,000 people?
- Will Trump deport 1,750,000-2,000,000 people?
- Will Trump deport 1,250,000-1,500,000 people?
- Will Trump deport 1,500,000-1,750,000 people?
- Will Trump deport 2,000,000 or more people?
- Will Trump deport 750,000 or more people in 2025?
- China x India military clash by December 31?
- China x India military clash by June 30?
- China x India military clash by December 31, 2026?
- NATO/EU troops fighting in Ukraine in 2025?
- NATO/EU troops fighting in Ukraine in June 30, 2026?
- Starmer out in 2025?
- Starmer out by June 30, 2026?

== us_china_semis_export_controls ==


,tag_id,label,slug,us_china_semis_export_controls,n_markets
0,2,Politics,politics,0,20
1,102169,Hide From New,hide-from-new,0,20
2,21,Crypto,crypto,0,20
3,101757,Recurring,recurring,0,20
4,1312,Crypto Prices,crypto-prices,0,20
5,102127,Up or Down,up-or-down,0,20
6,144,Elections,elections,0,20
7,102892,5M,5M,0,20
8,102786,Nov 4 Elections,nov-4-elections,0,20
9,103153,"rewards 5, 4.5, 50",rewards-5-4pt5-50,0,20


Best tag_id: 2 slug: politics
- Macron out in 2025?
- Macron out by June 30, 2026?
- Macron out by October 31, 2025?
- Will Trump deport less than 250,000?
- Will Trump deport 250,000-500,000 people?
- Will Trump deport 500,000-750,000- people?
- Will Trump deport 750,000-1,000,000 people?
- Will Trump deport 1,000,000-1,250,000 people?
- Will Trump deport 1,750,000-2,000,000 people?
- Will Trump deport 1,250,000-1,500,000 people?
- Will Trump deport 1,500,000-1,750,000 people?
- Will Trump deport 2,000,000 or more people?
- Will Trump deport 750,000 or more people in 2025?
- China x India military clash by December 31?
- China x India military clash by June 30?
- China x India military clash by December 31, 2026?
- NATO/EU troops fighting in Ukraine in 2025?
- NATO/EU troops fighting in Ukraine in June 30, 2026?
- Starmer out in 2025?
- Starmer out by June 30, 2026?
